In [12]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import os

In [31]:
# 1. temp view
for v in spark.catalog.listTables():
    if v.isTemporary:
        spark.catalog.dropTempView(v.name)

# 2. managed-таблицы
for t in spark.catalog.listTables():
    if not t.isTemporary:
        spark.sql(f"DROP TABLE IF EXISTS {t.name}")

# 3. кеши
spark.catalog.clearCache()

# 4. проверить


### Extract & Register: Postgres → Parquet with Optimizer Statistics

### Issue

The original data lives in plain PostgreSQL tables (and views). Reading tables 
straight over JDBC does not give
Spark's Catalyst optimizer any real statistics to work with — every JDBC
scan is treated as an unknown-size black box, forcing the planner to fall
back on defaults.

To let Spark make cost-based decisions (join strategy, join order,
filter selectivity), the tables and views are materialized once as
Parquet and registered in Spark's catalog, so that real statistics can
be computed and reused by every downstream query.

### Steps

1. **Discover source objects dynamically.**
   Query PostgreSQL's system catalog (`pg_class` / `pg_namespace`)
   instead of hardcoding table names, so the pipeline adapts
   automatically if the schema changes. Base tables, partition parent tables and views 
   are included, while partition child tables and
   unsupported object types (e.g. columns backed by `pgvector`) are
   explicitly excluded, since they would either duplicate data already
   present in the parent table or fail to read over JDBC.

2. **Materialize each object into Spark's managed catalog as Parquet.**
   Each table/view is read once via JDBC and written with
   `saveAsTable`, which stores the data as Parquet files under Spark's
   warehouse directory and registers the object in the catalog under
   the same name. From now on, everything is served from the local
   Parquet-backed catalog.

3. **Compute column-level statistics for supported types only.**
   `ANALYZE TABLE ... FOR COLUMNS` fails on complex types
   that PostgreSQL exposes for columns such
   as `film.special_features`. Scalar-typed columns are detected
   programmatically and passed explicitly to `ANALYZE`, so the
   statistics collection succeeds regardless of which complex-typed
   columns a given table happens to contain.

### Two independent layers of statistics

This pipeline produces two distinct kinds of statistics, at two
different layers, and both matter for query optimization:

- **Parquet file-level (footer) statistics** — min/max/null-count per
  column, computed automatically at write time, at row-group
  granularity. These require no `ANALYZE` step at all and are used
  directly by Spark's Parquet reader for physical row-group pruning
  (skipping blocks of data on disk that provably cannot match a given
  filter).

- **Catalog-level (Hive metastore) statistics** — row count, total size
  in bytes, and per-column min/max/distinct-count/null-count, populated
  only after running `ANALYZE TABLE ... COMPUTE STATISTICS FOR COLUMNS`.
  These are consumed by Spark's cost-based optimizer (CBO) during
  logical planning — deciding which side of a join to broadcast,
  reordering multi-way joins, and estimating filter selectivity before
  a single task is scheduled.

For this assignment's scale, file-level statistics alone would already
be sufficient for correct results; explicitly collecting catalog-level
statistics is what additionally lets the CBO make informed planning
decisions rather than falling back on defaults — which is closer i guess to how
this would be handled in a production pipeline reading from a
"live" transactional database.

In [ ]:
spark = (
    SparkSession.builder
    .appName('pagila-tasks')
    # JDBC-driver to load from PostgreSQL
    .config('spark.jars.packages', 'org.postgresql:postgresql:42.7.3')
    .config('spark.sql.shuffle.partitions', '4') #custom partitioning -> default is too much
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')
    # CBO
    .config('spark.sql.cbo.enabled', 'true')
    .config('spark.sql.cbo.joinReorder.enabled', 'true')
    # Metastore + warehouse
    .config('spark.sql.warehouse.dir', '/home/jovyan/work/spark-warehouse')
    .enableHiveSupport()      
    .getOrCreate()
)

In [ ]:
jdbc_url = f"jdbc:postgresql://db:5432/{os.environ['DB_NAME']}"
props = {
    'user': os.environ['DB_USER'],
    'password': os.environ['DB_PASSWORD'],
    'driver': 'org.postgresql.Driver',
}
#tables from pagila
tables_df = spark.read.jdbc(
    url=jdbc_url,
    table="""(
        SELECT c.relname AS tablename
        FROM pg_class c
        JOIN pg_namespace n ON n.oid = c.relnamespace
        WHERE n.nspname = 'public'
          AND c.relkind IN ('r', 'v', 'p')
          AND c.relname <> 'film_embedding'
          AND NOT c.relispartition
    ) AS t""",
    properties=props,
)
tables = [r['tablename'] for r in tables_df.collect()]

# dumping to warehouse
from pyspark.sql.types import ArrayType, MapType, StructType

def get_analyzable_columns(df):
    return [
        f"`{f.name}`" for f in df.schema.fields
        if not isinstance(f.dataType, (ArrayType, MapType, StructType))
    ]

for t in tables:
    df = spark.read.jdbc(url=jdbc_url, table=t, properties=props)
    df.write.mode("overwrite").saveAsTable(t)

    analyzable_cols = get_analyzable_columns(df)
    try:
        cols_str = ", ".join(analyzable_cols)
        spark.sql(f"ANALYZE TABLE `{t}` COMPUTE STATISTICS FOR COLUMNS {cols_str}")
    except Exception as e:
        print(f"ANALYZE FOR COLUMNS failed ({type(e).__name__}), fallback")
        spark.sql(f"ANALYZE TABLE `{t}` COMPUTE STATISTICS")


→ loading & analyzing film_actor
→ loading & analyzing address
→ loading & analyzing city
→ loading & analyzing actor
→ loading & analyzing inventory
→ loading & analyzing actor_info
→ loading & analyzing category
→ loading & analyzing country
→ loading & analyzing customer
→ loading & analyzing customer_list
→ loading & analyzing film_list
→ loading & analyzing film
→ loading & analyzing film_category
→ loading & analyzing nicer_but_slower_film_list
→ loading & analyzing language
→ loading & analyzing sales_by_film_category
→ loading & analyzing store
→ loading & analyzing sales_by_store
→ loading & analyzing staff_list
→ loading & analyzing payment
→ loading & analyzing rental
→ loading & analyzing staff


Output the number of movies in each category, sorted in descending order

In [33]:
film_category = spark.table("film_category")
category = spark.table("category")

res1 = (
    film_category.join(category.hint("broadcast"), "category_id")
    .groupBy("name")
    .agg(F.count("category_id").alias("movie_count"))
    .orderBy(F.desc("movie_count"))
)
res1.show()

+-----------+-----------+
|       name|movie_count|
+-----------+-----------+
|      Music|        152|
|      Drama|        152|
|     Travel|        151|
|    Foreign|        150|
|   Children|        150|
|      Games|        150|
|     Sci-Fi|        149|
|     Action|        149|
|  Animation|        148|
|        New|        147|
|     Family|        147|
|   Classics|        147|
|Documentary|        145|
|     Sports|        145|
|     Comedy|        143|
|     Horror|        142|
+-----------+-----------+



In [34]:
res1.explain("cost")   

== Optimized Logical Plan ==
Sort [movie_count#803858L DESC NULLS LAST], true, Statistics(sizeInBytes=560.0 B, rowCount=16)
+- Aggregate [name#803840], [name#803840, count(category_id#803834) AS movie_count#803858L], Statistics(sizeInBytes=560.0 B, rowCount=16)
   +- Project [category_id#803834, name#803840], Statistics(sizeInBytes=71.7 KiB, rowCount=2.37E+3)
      +- Join Inner, (category_id#803834 = category_id#803839), rightHint=(strategy=broadcast), Statistics(sizeInBytes=80.9 KiB, rowCount=2.37E+3)
         :- Project [category_id#803834], Statistics(sizeInBytes=27.7 KiB, rowCount=2.37E+3)
         :  +- Filter isnotnull(category_id#803834), Statistics(sizeInBytes=55.5 KiB, rowCount=2.37E+3)
         :     +- Relation spark_catalog.default.film_category[film_id#803833,category_id#803834,last_update#803835] parquet, Statistics(sizeInBytes=55.5 KiB, rowCount=2.37E+3)
         +- Project [category_id#803839, name#803840], Statistics(sizeInBytes=496.0 B, rowCount=16)
            +- Fi

Output the 10 actors whose movies rented the most, sorted in descending order. 

In [35]:
actor = spark.table("actor")
film_actor = spark.table("film_actor")
inventory = spark.table("inventory")
rental = spark.table("rental")

res2 = (
    rental
    .join(inventory, 'inventory_id')
    .join(film_actor, 'film_id')
    .groupBy('actor_id')
    .agg(F.count('*').alias('rental_num'))
    .join(actor, 'actor_id')
    .orderBy(F.desc('rental_num'))
    .limit(10)
)
res2.explain("cost")
res2.show()

== Optimized Logical Plan ==
GlobalLimit 10, Statistics(sizeInBytes=680.0 B, rowCount=10)
+- LocalLimit 10, Statistics(sizeInBytes=12.7 KiB, rowCount=200)
   +- Sort [rental_num#803954L DESC NULLS LAST], true, Statistics(sizeInBytes=12.7 KiB, rowCount=200)
      +- Project [actor_id#803883, rental_num#803954L, first_name#803876, last_name#803877, last_update#803878], Statistics(sizeInBytes=12.7 KiB, rowCount=200)
         +- Join Inner, (actor_id#803883 = actor_id#803875), Statistics(sizeInBytes=13.5 KiB, rowCount=200)
            :- Aggregate [actor_id#803883], [actor_id#803883, count(1) AS rental_num#803954L], Statistics(sizeInBytes=3.9 KiB, rowCount=199)
            :  +- Project [actor_id#803883], Statistics(sizeInBytes=3.1 MiB, rowCount=2.74E+5)
            :     +- Join Inner, (inventory_id#803899 = inventory_id#803889), Statistics(sizeInBytes=5.2 MiB, rowCount=2.74E+5)
            :        :- Project [inventory_id#803889, actor_id#803883], Statistics(sizeInBytes=381.1 KiB, rowCo

Output the category of movies on which the most money was spent

In [37]:
sales_by_film_category = spark.table("sales_by_film_category")

res3 = (
    sales_by_film_category
    .withColumn('rnk', F.rank().over(Window.orderBy(F.desc('total_sales'))))
    .filter(F.col('rnk') == 1)
    .select('category', F.round('total_sales', 2).alias('total_sales'))
)
res3.show()

+--------+-----------+
|category|total_sales|
+--------+-----------+
|  Action|   26505.44|
+--------+-----------+



In [ ]:
res3.explain('formatted')

Output the names of movies that are not in the inventory.

In [ ]:
film = spark.table("film")
inv_ids = spark.table("inventory").select("film_id") # explicitly
res4 = (
    film.join(inv_ids, 'film_id', 'left_anti').select("title")
)
res4.show()

Output the top 3 actors who have appeared most in movies in the “Children” category. If several actors have the same number of movies, output all of them. 

In [ ]:
actor = spark.table("actor")
film_actor = spark.table("film_actor")
film_category = spark.table("film_category")
category = spark.table("category")

#  category_id scalar ->.collect() 
children_id = (
    category.filter(F.col("name") == "Children")
    .select("category_id")
    .collect()[0]["category_id"]
)

actor_children = (
    film_actor
    .join(film_category.filter(F.col("category_id") == children_id), "film_id")  # фильтр до джойна
    .groupBy("actor_id")
    .agg(F.count("*").alias("sum_of_appearance"))
    .join(actor, "actor_id")  
)
actor_children.cache() 

w = Window.orderBy(F.desc("sum_of_appearance"))
res5 = (
    actor_children
    .withColumn("rnk", F.rank().over(w))
    .filter(F.col("rnk") <= 3)
    .select("actor_id", "first_name", "last_name", "sum_of_appearance", "rnk")
    .orderBy("rnk")
)
res5.explain("cost")
res5.show()

actor_children.unpersist()

Output cities with the number of active and inactive customers (active - customer.active = 1). Sort by the number of inactive customers in descending order

In [38]:
customer = spark.table("customer")
city = spark.table("city")
address = spark.table("address")

customer_addres = (
    customer
    .join(address, "address_id") 
    .join(city, "city_id") 
    .groupBy("city_id", "city")
    .agg(
        F.sum(F.when(F.col("active") == 1, 1).otherwise(0)).alias("active_count"),
        F.sum(F.when(F.col("active") != 1, 1).otherwise(0)).alias("inactive_count"),
    )
    .orderBy(F.desc("inactive_count"))  
)
customer_addres.show()

+-------+--------------------+------------+--------------+
|city_id|                city|active_count|inactive_count|
+-------+--------------------+------------+--------------+
|    452|San Juan Bautista...|         383|            18|
|    495|     Southend-on-Sea|           0|             1|
|    578|            Xiangfan|           0|             1|
|    283|          Kumbakonam|           0|             1|
|    281|              Ktahya|           0|             1|
|     57|             Bat Yam|           0|             1|
|    356|           Najafabad|           0|             1|
|    111|    Charlotte Amalie|           0|             1|
|    125|       Coatzacoalcos|           0|             1|
|    577|             Wroclaw|           0|             1|
|    512|         Szkesfehrvr|           0|             1|
|    139|              Daxian|           0|             1|
|    407|           Pingxiang|           0|             1|
|    554|            Uluberia|           0|             

Output the category of movies that have the highest number of total rental hours in the cities (customer.address_id in this city), and that start with the letter “a”. Do the same for cities with a “-” symbol.

In [45]:
rental = (
    spark.table("rental")
    .filter(F.col("return_date").isNotNull())
    .withColumn("rental_hours",
        (F.col("return_date").cast("long") - F.col("rental_date").cast("long")) / 3600.0)
)

city_filtered = spark.table("city").filter(
    F.lower(F.col("city")).startswith("a") | F.col("city").contains("-")
)

rental_hours = (
    rental
    .join(customer, "customer_id")
    .join(address, "address_id")
    .join(city_filtered.hint("broadcast"), "city_id") 
    .join(inventory, "inventory_id")
    .join(film_category, "film_id")
    .join(category.hint("broadcast"), "category_id")
    .groupBy("city_id", "city", "name")
    .agg(F.sum("rental_hours").alias("hours"))
)
rental_hours.cache()

grouped = (
    rental_hours
    .filter(F.lower(F.col("city")).startswith("a"))
    .withColumn("city_group", F.lit("starts_with_a"))
    .unionAll(
        rental_hours
        .filter(F.col("city").contains("-"))
        .withColumn("city_group", F.lit("contains_dash"))
    )
)

w = Window.partitionBy("city_id", "city_group").orderBy(F.desc("hours"))
result = (
    grouped
    .withColumn("rnk", F.rank().over(w))
    .filter(F.col("rnk") == 1)
    .select(
        "city_id", "city", "city_group", "name",
        F.round("hours", 2).alias("total_hours")
    )
    .orderBy(
        F.when(F.col("city_group") == "starts_with_a", 1).otherwise(2),
        "city", "city_id",
    )
)
result.show()

rental_hours.unpersist()

+-------+--------------------+-------------+-----------+-----------+
|city_id|                city|   city_group|       name|total_hours|
+-------+--------------------+-------------+-----------+-----------+
|      1|  A Corua (La Corua)|starts_with_a|     Family|    1977.69|
|      2|                Abha|starts_with_a|     Action|    2067.79|
|      3|           Abu Dhabi|starts_with_a|        New|    2127.64|
|      4|                Acua|starts_with_a|   Children|    1764.62|
|      5|               Adana|starts_with_a|     Sci-Fi|    2361.39|
|      6|         Addis Abeba|starts_with_a|     Action|    1949.65|
|      7|                Aden|starts_with_a|        New|    2072.07|
|      8|               Adoni|starts_with_a|      Games|     2398.3|
|      9|          Ahmadnagar|starts_with_a|    Foreign|    1801.81|
|     10|            Akishima|starts_with_a|   Children|    2379.45|
|     11|               Akron|starts_with_a|     Horror|    2082.15|
|     17|         Alessandria|star

DataFrame[city_id: int, city: string, name: string, hours: double]